
# Extended PCA vs ICA vs MS-AA network-motif simulation benchmark

This notebook keeps the **simple ICA policy** from the previous benchmark:
one deterministic FastICA fit per dataset, all ICA results retained, and
convergence reported only as a diagnostic.

It adds the remaining checks planned for the manuscript revision:

1. **Method calibration** with PCA-, ICA-, and AA-favorable controls.
2. **Brain-inspired metastable dynamics** that are not generated exactly by
   PCA, ICA, or AA.
3. **Motif-overlap sensitivity** (low / baseline / high overlap).
4. **Dynamics sensitivity** (state-like / baseline / more continuously mixed).
5. **Temporal latent-activation recovery** in addition to spatial motif recovery.
6. **Normalization QA** against the exact empirical `hypertools/normalize.py`
   routine when available.
7. Existing noise, subject-heterogeneity, and fitted-\(K\) robustness checks.

### Two technical fixes are also incorporated

- The metastable generator is now deliberately **non-simplex-exact**:
  after state persistence, gradual transitions, and secondary recruitment,
  the latent activations receive time-varying global gain and small signed
  autocorrelated deviations. They are not constrained to be nonnegative or
  to sum to one.
- PCA/ICA generative **directions** are transformed differently from
  AA/metastable **state points** after centering/scaling.

Temporal recovery is anchored to the **spatial matching**, so the temporal
question is literally: after identifying which recovered dimension corresponds
to a true configuration, did that same dimension recover *when* the
configuration was expressed?

This notebook uses a **new checkpoint directory** because the metastable
generator and normalization logic changed.

### Revision 14 note

The normalization QA now loads the empirical `hypertools/normalize.py`
together with its local module directory, so imports such as
`format_data` and `helpers` resolve correctly. The configuration cell also
imports NumPy explicitly so the notebook runs cleanly from a fresh kernel.

### Revision 15 note

The normalization QA now calls the empirical HyperTools normalizer with
`format_data=False` because the QA inputs have already been coerced to numeric
T × V arrays. This bypasses unrelated legacy formatting dependencies
(`df2mat`, `text2mat`, `datageometry`) while testing the exact normalization
logic itself.


In [ ]:
import numpy as np


# ============================================================
# Configuration
# ============================================================

RUN_MODE = "quick"   # "quick" or "paper"
RNG_SEED = 42

HELPERS_DIR = "helpers"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"
OUTPUT_DIR = "simulation_pca_ica_msaa_extended_outputs"

ANALYSIS_TYPE = "spatial"

MSAA_OPTS_BASE = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

# Shared spatial architecture
N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10
N_TRUE_MOTIFS = 6

BASE_NOISE_SD = 0.25
BASE_MULTIPLICATIVE_SD = 0.08
BASE_ADDITIVE_SD = 0.04

# PCA-positive-control source standard deviations.
PCA_SOURCE_SCALES = np.array(
    [2.0, 1.7, 1.4, 1.1, 0.8, 0.5]
)

# AA-positive-control geometry
DIRICHLET_ALPHA = 0.10

# Brain-inspired metastable dynamics
MEAN_DWELL = 12
TRANSITION_WIDTH = 4
SECONDARY_WEIGHT = 0.25
AR1_RHO = 0.85

# Make metastable dynamics non-simplex-exact.
GLOBAL_GAIN_SD = 0.15
SIGNED_DEVIATION_SD = 0.08
DEVIATION_RHO = 0.80

# Motif overlap manipulation.
# Higher distinctiveness -> lower overlap.
OVERLAP_LEVELS = {
    "low_overlap": 1.15,
    "baseline_overlap": 1.00,
    "high_overlap": 0.55,
}

# Joint dynamics manipulation.
DYNAMICS_PRESETS = {
    "state_like": {
        "mean_dwell": 16,
        "transition_width": 2,
        "secondary_weight": 0.10,
        "ar1_rho": 0.75,
    },
    "baseline": {
        "mean_dwell": 12,
        "transition_width": 4,
        "secondary_weight": 0.25,
        "ar1_rho": 0.85,
    },
    "more_mixed": {
        "mean_dwell": 8,
        "transition_width": 8,
        "secondary_weight": 0.45,
        "ar1_rho": 0.90,
    },
}

# Secondary robustness
NOISE_LEVELS = [0.10, 0.25, 0.40]
HETEROGENEITY_LEVELS = [0.0, 1.0, 1.5]
K_LEVELS = [4, 6, 8]

if RUN_MODE == "quick":
    N_SUBJECTS = 10
    T_PER_SUBJECT = 120
    N_REPS_CORE = 2
    N_REPS_METASTABLE = 3
    N_REPS_ROBUST = 1
    N_REPS_SENSITIVITY = 1
elif RUN_MODE == "paper":
    N_SUBJECTS = 16
    T_PER_SUBJECT = 180
    N_REPS_CORE = 20
    N_REPS_METASTABLE = 20
    N_REPS_ROBUST = 10
    N_REPS_SENSITIVITY = 10
else:
    raise ValueError(
        "RUN_MODE must be 'quick' or 'paper'"
    )

print("RUN_MODE:", RUN_MODE)
print("Subjects:", N_SUBJECTS)
print("Time points per subject:", T_PER_SUBJECT)
print("Sensitivity reps:", N_REPS_SENSITIVITY)


In [ ]:

# ============================================================
# Imports
# ============================================================

%matplotlib inline

import os
import sys
import time
import warnings
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA, FastICA
from sklearn.exceptions import ConvergenceWarning

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_CSV = OUTPUT_DIR / (
    f"pca_ica_msaa_extended_results_{RUN_MODE}.csv"
)

print("Checkpoint:", RESULTS_CSV)


## 1. MS-AA helpers matching the empirical pipeline

In [ ]:

def coerce_TxV(x):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D T x V matrix, got {x.shape}")

    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)


def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)
    return Xtv if analysis_type == "spatial" else Xtv.T


def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]


def fit_msaa(subject_list, K, analysis_type="spatial", seed=RNG_SEED, verbose=False):
    opts = dict(MSAA_OPTS_BASE)
    opts["rngSEED"] = int(seed)

    subj_objs = build_subject_objects(subject_list, analysis_type)

    t0 = time.time()

    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=int(K),
        opts=opts,
    )

    elapsed = time.time() - t0

    if verbose:
        print(
            f"[MS-AA] K={K} | VarExpl={varexpl*100:.2f}% | "
            f"{elapsed:.2f}s"
        )

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "elapsed": elapsed,
        "K": int(K),
    }


def reconstruct_from_archetypes(sub, analysis_type="spatial"):
    sXC = np.asarray(sub["sXC"], dtype=float)
    S = np.asarray(sub["S"], dtype=float)

    if analysis_type == "spatial":
        return sXC @ S

    return (sXC @ S).T


## 2. Network-motif generator

In [ ]:

def make_overlapping_motifs():
    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    V = (
        n_subnetworks
        * NODES_PER_SUBNETWORK
    )

    motifs = np.full(
        (N_TRUE_MOTIFS, V),
        0.05,
        dtype=float,
    )

    motif_subnetworks = [
        [0, 3, 8],
        [1, 3, 9],
        [2, 5, 10],
        [2, 4, 11],
        [0, 6, 7],
        [1, 6, 8],
    ]

    for k, active in enumerate(
        motif_subnetworks
    ):
        for subnet in active:
            lo = (
                subnet
                * NODES_PER_SUBNETWORK
            )
            hi = (
                lo
                + NODES_PER_SUBNETWORK
            )

            motifs[
                k,
                lo:hi,
            ] = 1.0

    return motifs


def make_overlap_variant(
    base_motifs,
    distinctiveness_scale,
):
    # Scale each motif's deviation from the common motif.
    # This preserves network topology while changing
    # nonorthogonality/overlap.
    base_motifs = np.asarray(
        base_motifs,
        dtype=float,
    )

    common = base_motifs.mean(
        axis=0,
        keepdims=True,
    )

    return (
        common
        + distinctiveness_scale
        * (
            base_motifs
            - common
        )
    )


def mean_pairwise_cosine(patterns):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    unit = patterns / (
        np.linalg.norm(
            patterns,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = unit @ unit.T

    tri = np.triu_indices(
        C.shape[0],
        k=1,
    )

    return float(
        np.mean(C[tri])
    )


TRUE_MOTIFS_RAW = (
    make_overlapping_motifs()
)

V = TRUE_MOTIFS_RAW.shape[1]

print(
    "Motif matrix:",
    TRUE_MOTIFS_RAW.shape,
)

for name, scale in (
    OVERLAP_LEVELS.items()
):
    variant = make_overlap_variant(
        TRUE_MOTIFS_RAW,
        scale,
    )

    print(
        name,
        "| distinctiveness =",
        scale,
        "| mean pairwise cosine =",
        round(
            mean_pairwise_cosine(
                variant
            ),
            3,
        ),
    )


In [ ]:

def plot_pattern_matrix(patterns, title, ylabel="Pattern"):
    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(patterns, aspect="auto")

    for x in range(
        NODES_PER_SUBNETWORK,
        patterns.shape[1],
        NODES_PER_SUBNETWORK,
    ):
        ax.axvline(x - 0.5, linewidth=0.4, alpha=0.35)

    network_width = (
        SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    for x in range(
        network_width,
        patterns.shape[1],
        network_width,
    ):
        ax.axvline(x - 0.5, linewidth=1.2)

    ax.set_xlabel("Nodes grouped by network → subnetwork")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.show()


plot_pattern_matrix(
    TRUE_MOTIFS_RAW,
    "Ground-truth overlapping distributed network motifs",
    ylabel="True motif",
)


## 3. Common normalization and evaluation metrics

In [ ]:

def normalize_across_subjects(
    subject_list,
    eps=1e-8,
):
    # Match the empirical fallback:
    # coerce/center EACH subject first,
    # then scale features across the stacked subjects.
    raw = [
        np.asarray(
            x,
            dtype=float,
        )
        for x in subject_list
    ]

    subject_means = [
        x.mean(
            axis=0,
            keepdims=True,
        )
        for x in raw
    ]

    centered = [
        x - mu_s
        for x, mu_s in zip(
            raw,
            subject_means,
        )
    ]

    stacked = np.vstack(
        centered
    )

    across_mu = stacked.mean(
        axis=0,
        keepdims=True,
    )

    sd = (
        stacked.std(
            axis=0,
            keepdims=True,
        )
        + eps
    )

    normalized = [
        (
            (x - across_mu)
            / sd
        ).astype(
            np.float32,
            copy=False,
        )
        for x in centered
    ]

    mean_subject_mean = np.mean(
        np.stack(
            subject_means,
            axis=0,
        ),
        axis=0,
    )

    return (
        normalized,
        mean_subject_mean,
        sd,
    )


def transform_true_patterns(
    true_patterns_raw,
    mean_subject_mean,
    sd,
    truth_kind,
):
    true_patterns_raw = np.asarray(
        true_patterns_raw,
        dtype=float,
    )

    if truth_kind == "direction":
        # Centering observations changes origins,
        # not generative directions.
        return (
            true_patterns_raw
            / sd
        )

    if truth_kind == "point":
        # State/archetype templates are points.
        # Averaging subject-wise affine transforms
        # gives this group-level transform.
        return (
            (
                true_patterns_raw
                - mean_subject_mean
            )
            / sd
        )

    raise ValueError(
        "truth_kind must be "
        "'direction' or 'point'"
    )


def pattern_corr_matrix(
    A,
    B,
    absolute=False,
):
    A = np.asarray(
        A,
        dtype=float,
    )

    B = np.asarray(
        B,
        dtype=float,
    )

    A0 = (
        A
        - A.mean(
            axis=1,
            keepdims=True,
        )
    )

    B0 = (
        B
        - B.mean(
            axis=1,
            keepdims=True,
        )
    )

    A0 /= (
        np.linalg.norm(
            A0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    B0 /= (
        np.linalg.norm(
            B0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = A0 @ B0.T

    return (
        np.abs(C)
        if absolute
        else C
    )


def optimal_match(
    true_patterns,
    recovered_patterns,
    absolute=False,
):
    C = pattern_corr_matrix(
        true_patterns,
        recovered_patterns,
        absolute=absolute,
    )

    (
        true_idx,
        recovered_idx,
    ) = linear_sum_assignment(
        -C
    )

    return {
        "similarity": C,
        "true_idx": true_idx,
        "recovered_idx": recovered_idx,
        "matched": C[
            true_idx,
            recovered_idx,
        ],
    }


def coverage_adjusted_recovery(
    match,
    n_true,
):
    return float(
        np.sum(
            match["matched"]
        )
        / n_true
    )



def temporal_recovery_from_spatial_match(
    true_activations,
    recovered_scores,
    spatial_match,
    absolute=False,
):
    # Temporal recovery is anchored to spatial matching.
    # Because simulated observations are centered within subject,
    # center latent/recovered trajectories within subject before
    # concatenating across subjects.
    if true_activations is None:
        return {
            "mean": np.nan,
            "coverage": np.nan,
            "per_pattern": (
                np.array([])
            ),
        }

    true_list = [
        np.asarray(
            a,
            dtype=float,
        )
        for a in true_activations
    ]

    true_centered = [
        a - a.mean(
            axis=0,
            keepdims=True,
        )
        for a in true_list
    ]

    true_stack = np.vstack(
        true_centered
    )

    if isinstance(
        recovered_scores,
        (list, tuple),
    ):
        recovered_list = [
            np.asarray(
                a,
                dtype=float,
            )
            for a in recovered_scores
        ]
    else:
        recovered_array = np.asarray(
            recovered_scores,
            dtype=float,
        )

        lengths = [
            a.shape[0]
            for a in true_list
        ]

        cuts = np.cumsum(
            lengths
        )[:-1]

        recovered_list = (
            np.split(
                recovered_array,
                cuts,
                axis=0,
            )
        )

    recovered_centered = [
        a - a.mean(
            axis=0,
            keepdims=True,
        )
        for a in recovered_list
    ]

    recovered_stack = np.vstack(
        recovered_centered
    )

    vals = []

    for (
        true_idx,
        recovered_idx,
    ) in zip(
        spatial_match["true_idx"],
        spatial_match[
            "recovered_idx"
        ],
    ):
        a = true_stack[
            :,
            true_idx,
        ]

        b = recovered_stack[
            :,
            recovered_idx,
        ]

        if (
            np.std(a) < 1e-12
            or np.std(b) < 1e-12
        ):
            r = np.nan
        else:
            r = np.corrcoef(
                a,
                b,
            )[0, 1]

        if (
            absolute
            and np.isfinite(r)
        ):
            r = abs(r)

        vals.append(r)

    vals = np.asarray(
        vals,
        dtype=float,
    )

    return {
        "mean": float(
            np.nanmean(vals)
        ),
        "coverage": float(
            np.nansum(vals)
            / true_stack.shape[1]
        ),
        "per_pattern": vals,
    }

def collapse_to_subnetworks(
    patterns,
):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    out = np.zeros(
        (
            patterns.shape[0],
            n_subnetworks,
        )
    )

    for subnet in range(
        n_subnetworks
    ):
        lo = (
            subnet
            * NODES_PER_SUBNETWORK
        )

        hi = (
            lo
            + NODES_PER_SUBNETWORK
        )

        out[
            :,
            subnet,
        ] = (
            patterns[
                :,
                lo:hi,
            ].mean(
                axis=1
            )
        )

    return out


def dimension_mixing_metrics(
    true_patterns,
    recovered_patterns,
):
    C = pattern_corr_matrix(
        recovered_patterns,
        true_patterns,
        absolute=True,
    )

    squared = C ** 2

    probs = squared / (
        squared.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    entropy = -np.sum(
        probs
        * np.log(
            probs + 1e-12
        ),
        axis=1,
    )

    entropy /= np.log(
        true_patterns.shape[0]
    )

    purity = (
        1.0
        - entropy
    )

    sorted_corr = np.sort(
        C,
        axis=1,
    )[:, ::-1]

    top1 = sorted_corr[:, 0]

    top2 = (
        sorted_corr[:, 1]
        if (
            sorted_corr.shape[1]
            > 1
        )
        else np.zeros_like(
            top1
        )
    )

    return {
        "purity": float(
            np.mean(purity)
        ),
        "selectivity": float(
            np.mean(
                top1
                - top2
            )
        ),
        "mixing_ratio": float(
            np.mean(
                top2
                / (
                    top1
                    + 1e-12
                )
            )
        ),
    }


def linear_span_recovery(
    true_patterns,
    recovered_patterns,
):
    # Project true patterns onto the row span
    # of the recovered patterns.
    B = np.asarray(
        recovered_patterns,
        dtype=float,
    ).T

    vals = []

    for true in np.asarray(
        true_patterns,
        dtype=float,
    ):
        coef, *_ = np.linalg.lstsq(
            B,
            true,
            rcond=None,
        )

        recon = (
            B @ coef
        )

        vals.append(
            np.corrcoef(
                true,
                recon,
            )[0, 1]
        )

    return float(
        np.nanmean(vals)
    )



## 4. Unified PCA / ICA / MS-AA evaluator

The direct comparison is between **spatial patterns**:

- PCA: `pca.components_`
- ICA: `ica.mixing_.T`
- MS-AA: group-averaged `S`

PCA and ICA pattern correlations are sign-invariant. MS-AA uses signed correlations because its archetypes are not sign-indeterminate in the same way.

For fitted \(K < K_{\mathrm{true}}\), the notebook also reports a **coverage-adjusted recovery** score so that recovering only a subset of true motifs cannot look artificially strong.


In [ ]:

def fit_fastica(
    X_stack,
    K_fit,
    seed,
):
    ica = FastICA(
        n_components=int(K_fit),
        whiten="unit-variance",
        algorithm="parallel",
        fun="logcosh",
        max_iter=3000,
        tol=1e-5,
        random_state=int(seed),
    )

    with warnings.catch_warnings(
        record=True
    ) as caught:
        warnings.simplefilter(
            "always",
            ConvergenceWarning,
        )

        Z = ica.fit_transform(
            X_stack
        )

        Xhat = (
            ica.inverse_transform(
                Z
            )
        )

    converged = not any(
        issubclass(
            w.category,
            ConvergenceWarning,
        )
        for w in caught
    )

    spatial_patterns = (
        np.asarray(
            ica.mixing_,
            dtype=float,
        ).T
    )

    return {
        "model": ica,
        "scores": Z,
        "reconstruction": Xhat,
        "spatial_patterns": (
            spatial_patterns
        ),
        "converged": bool(
            converged
        ),
        "n_iter": int(
            getattr(
                ica,
                "n_iter_",
                -1,
            )
        ),
    }


def evaluate_methods(
    subject_data,
    true_patterns_raw,
    K_fit,
    seed,
    regime,
    rep,
    true_activations=None,
    truth_kind="point",
    extra=None,
    verbose=False,
):
    (
        subjects,
        mean_subject_mean,
        sd,
    ) = (
        normalize_across_subjects(
            subject_data
        )
    )

    true_patterns = (
        transform_true_patterns(
            true_patterns_raw,
            mean_subject_mean,
            sd,
            truth_kind=truth_kind,
        )
    )

    X_stack = np.vstack(
        subjects
    )

    # ========================================================
    # MS-AA
    # ========================================================

    msaa = fit_msaa(
        subjects,
        K=K_fit,
        analysis_type="spatial",
        seed=seed,
        verbose=verbose,
    )

    S_group = np.mean(
        np.stack(
            [
                np.asarray(
                    sub["S"],
                    dtype=float,
                )
                for sub in (
                    msaa[
                        "results_subj"
                    ]
                )
            ],
            axis=0,
        ),
        axis=0,
    )

    aa_scores = [
        np.asarray(
            sub["sXC"],
            dtype=float,
        )
        for sub in (
            msaa[
                "results_subj"
            ]
        )
    ]

    Xhat_aa = np.vstack([
        reconstruct_from_archetypes(
            sub,
            "spatial",
        )
        for sub in (
            msaa[
                "results_subj"
            ]
        )
    ])

    aa_mse = float(
        np.mean(
            (
                X_stack
                - Xhat_aa
            ) ** 2
        )
    )

    # ========================================================
    # PCA
    # ========================================================

    pca = PCA(
        n_components=int(K_fit),
        random_state=int(seed),
    )

    pca_scores = (
        pca.fit_transform(
            X_stack
        )
    )

    Xhat_pca = (
        pca.inverse_transform(
            pca_scores
        )
    )

    pca_patterns = np.asarray(
        pca.components_,
        dtype=float,
    )

    pca_mse = float(
        np.mean(
            (
                X_stack
                - Xhat_pca
            ) ** 2
        )
    )

    # ========================================================
    # ICA
    # ========================================================

    ica_fit = fit_fastica(
        X_stack,
        K_fit,
        seed,
    )

    ica_patterns = (
        ica_fit[
            "spatial_patterns"
        ]
    )

    ica_scores = (
        ica_fit["scores"]
    )

    Xhat_ica = (
        ica_fit[
            "reconstruction"
        ]
    )

    ica_mse = float(
        np.mean(
            (
                X_stack
                - Xhat_ica
            ) ** 2
        )
    )

    # ========================================================
    # Spatial recovery
    # ========================================================

    aa_match = optimal_match(
        true_patterns,
        S_group,
        absolute=False,
    )

    pca_match = optimal_match(
        true_patterns,
        pca_patterns,
        absolute=True,
    )

    ica_match = optimal_match(
        true_patterns,
        ica_patterns,
        absolute=True,
    )

    # ========================================================
    # Temporal recovery anchored to spatial match
    # ========================================================

    aa_temporal = (
        temporal_recovery_from_spatial_match(
            true_activations,
            aa_scores,
            aa_match,
            absolute=False,
        )
    )

    pca_temporal = (
        temporal_recovery_from_spatial_match(
            true_activations,
            pca_scores,
            pca_match,
            absolute=True,
        )
    )

    ica_temporal = (
        temporal_recovery_from_spatial_match(
            true_activations,
            ica_scores,
            ica_match,
            absolute=True,
        )
    )

    # ========================================================
    # Subnetwork recovery
    # ========================================================

    true_sub = (
        collapse_to_subnetworks(
            true_patterns
        )
    )

    aa_sub = (
        collapse_to_subnetworks(
            S_group
        )
    )

    pca_sub = (
        collapse_to_subnetworks(
            pca_patterns
        )
    )

    ica_sub = (
        collapse_to_subnetworks(
            ica_patterns
        )
    )

    aa_sub_match = optimal_match(
        true_sub,
        aa_sub,
        absolute=False,
    )

    pca_sub_match = optimal_match(
        true_sub,
        pca_sub,
        absolute=True,
    )

    ica_sub_match = optimal_match(
        true_sub,
        ica_sub,
        absolute=True,
    )

    # ========================================================
    # Span recovery
    # ========================================================

    aa_span = linear_span_recovery(
        true_patterns,
        S_group,
    )

    pca_span = linear_span_recovery(
        true_patterns,
        pca_patterns,
    )

    ica_span = linear_span_recovery(
        true_patterns,
        ica_patterns,
    )

    # ========================================================
    # Secondary mixing diagnostics
    # ========================================================

    aa_mix = (
        dimension_mixing_metrics(
            true_patterns,
            S_group,
        )
    )

    pca_mix = (
        dimension_mixing_metrics(
            true_patterns,
            pca_patterns,
        )
    )

    ica_mix = (
        dimension_mixing_metrics(
            true_patterns,
            ica_patterns,
        )
    )

    n_true = (
        true_patterns.shape[0]
    )

    row = {
        "regime": regime,
        "condition": "base",
        "rep": int(rep),
        "seed": int(seed),
        "K_fit": int(K_fit),
        "truth_kind": truth_kind,

        "aa_motif_recovery": float(
            np.mean(
                aa_match["matched"]
            )
        ),
        "pca_motif_recovery": float(
            np.mean(
                pca_match["matched"]
            )
        ),
        "ica_motif_recovery": float(
            np.mean(
                ica_match["matched"]
            )
        ),

        "aa_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                aa_match,
                n_true,
            )
        ),
        "pca_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                pca_match,
                n_true,
            )
        ),
        "ica_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                ica_match,
                n_true,
            )
        ),

        "aa_temporal_recovery": (
            aa_temporal["mean"]
        ),
        "pca_temporal_recovery": (
            pca_temporal["mean"]
        ),
        "ica_temporal_recovery": (
            ica_temporal["mean"]
        ),

        "aa_temporal_recovery_coverage": (
            aa_temporal[
                "coverage"
            ]
        ),
        "pca_temporal_recovery_coverage": (
            pca_temporal[
                "coverage"
            ]
        ),
        "ica_temporal_recovery_coverage": (
            ica_temporal[
                "coverage"
            ]
        ),

        "aa_subnetwork_recovery": float(
            np.mean(
                aa_sub_match[
                    "matched"
                ]
            )
        ),
        "pca_subnetwork_recovery": float(
            np.mean(
                pca_sub_match[
                    "matched"
                ]
            )
        ),
        "ica_subnetwork_recovery": float(
            np.mean(
                ica_sub_match[
                    "matched"
                ]
            )
        ),

        "aa_span_recovery": aa_span,
        "pca_span_recovery": pca_span,
        "ica_span_recovery": ica_span,

        "aa_mse": aa_mse,
        "pca_mse": pca_mse,
        "ica_mse": ica_mse,

        "aa_purity": (
            aa_mix["purity"]
        ),
        "pca_purity": (
            pca_mix["purity"]
        ),
        "ica_purity": (
            ica_mix["purity"]
        ),

        "aa_selectivity": (
            aa_mix["selectivity"]
        ),
        "pca_selectivity": (
            pca_mix["selectivity"]
        ),
        "ica_selectivity": (
            ica_mix["selectivity"]
        ),

        "aa_mixing_ratio": (
            aa_mix["mixing_ratio"]
        ),
        "pca_mixing_ratio": (
            pca_mix["mixing_ratio"]
        ),
        "ica_mixing_ratio": (
            ica_mix["mixing_ratio"]
        ),

        "aa_varexpl": float(
            msaa["varexpl"]
        ),
        "pca_explained_variance": float(
            np.sum(
                pca.explained_variance_ratio_
            )
        ),

        # ICA convergence is diagnostic only.
        "ica_converged": bool(
            ica_fit["converged"]
        ),
        "ica_n_iter": int(
            ica_fit["n_iter"]
        ),

        "msaa_elapsed_sec": float(
            msaa["elapsed"]
        ),
    }

    if extra:
        row.update(extra)

    return row



# REGIME 1 — PCA-favorable control

The true spatial factors are orthogonal. Their temporal scores are independent Gaussian variables with **distinct variances**, making the generating PCA axes identifiable in the population covariance.

PCA should be the strongest direct-pattern method here.

ICA has no unique independent-component solution for Gaussian sources, while MS-AA is being asked to represent a variance-axis geometry that is not archetypal.


In [ ]:

def simulate_pca_control(seed):
    rng = np.random.default_rng(
        seed
    )

    Q, _ = np.linalg.qr(
        rng.normal(
            size=(
                V,
                N_TRUE_MOTIFS,
            )
        )
    )

    true_factors = Q.T

    subjects = []
    activations = []

    for s in range(
        N_SUBJECTS
    ):
        scores = rng.normal(
            size=(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
            )
        )

        scores = (
            scores
            * PCA_SOURCE_SCALES[
                None,
                :,
            ]
        )

        X = (
            scores
            @ true_factors
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)
        activations.append(
            scores
        )

    return {
        "subjects": subjects,
        "true_patterns": (
            true_factors
        ),
        "activations": (
            activations
        ),
        "truth_kind": "direction",
    }



# REGIME 2 — ICA-favorable control

The true spatial patterns are dense and nonorthogonal. Their temporal sources are statistically independent and strongly non-Gaussian (Laplace), and observations are generated by a linear mixing model.

ICA should recover these spatial mixing patterns most directly.

PCA can capture their low-dimensional span but is not expected to identify the independent mixing vectors. MS-AA is not being given an extremal/simplex geometry.


In [ ]:

def make_ica_mixing_patterns(
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    base = rng.normal(
        size=(
            N_TRUE_MOTIFS,
            V,
        )
    )

    shared = rng.normal(
        size=(1, V)
    )

    shared /= (
        np.linalg.norm(shared)
        + 1e-12
    )

    patterns = (
        base
        + 0.45 * shared
    )

    patterns -= (
        patterns.mean(
            axis=1,
            keepdims=True,
        )
    )

    patterns /= (
        np.linalg.norm(
            patterns,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return patterns


def simulate_ica_control(
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    true_patterns = (
        make_ica_mixing_patterns(
            seed + 999
        )
    )

    subjects = []
    activations = []

    for s in range(
        N_SUBJECTS
    ):
        sources = rng.laplace(
            loc=0.0,
            scale=(
                1.0
                / np.sqrt(2.0)
            ),
            size=(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
            ),
        )

        X = (
            sources
            @ true_patterns
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)
        activations.append(
            sources
        )

    return {
        "subjects": subjects,
        "true_patterns": (
            true_patterns
        ),
        "activations": (
            activations
        ),
        "truth_kind": "direction",
    }



# REGIME 3 — AA-favorable control

Each observation is a sparse convex mixture of nonorthogonal extremal motifs, with mild subject-specific perturbations.

MS-AA should recover the generating motifs most directly.

PCA and ICA may still preserve much of the low-dimensional span, but their individual components need not correspond to the extremal generators.


In [ ]:

def simulate_aa_control(
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    subjects = []
    activations = []

    for s in range(
        N_SUBJECTS
    ):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD,
                size=(
                    TRUE_MOTIFS_RAW.shape
                ),
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD,
            size=(
                TRUE_MOTIFS_RAW.shape
            ),
        )

        subj_motifs = (
            TRUE_MOTIFS_RAW
            * multiplicative
            + additive
        )

        W = rng.dirichlet(
            np.full(
                N_TRUE_MOTIFS,
                DIRICHLET_ALPHA,
            ),
            size=T_PER_SUBJECT,
        )

        X = (
            W @ subj_motifs
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)
        activations.append(W)

    return {
        "subjects": subjects,
        "true_patterns": (
            TRUE_MOTIFS_RAW
        ),
        "activations": (
            activations
        ),
        "truth_kind": "point",
    }



# REGIME 4 — Brain-inspired metastable network dynamics

This is deliberately **not** a PCA, ICA, or AA generative model.

Network configurations recur over time, remain active for variable dwell
periods, transition gradually, partially coactivate, and vary across subjects.
Background motif recruitment is temporally autocorrelated.

After these state-like dynamics are generated, the latent activations receive
time-varying global gain and small signed autocorrelated deviations:

\[
X_t = \sum_k A^{\ast}_{tk}M_k^{(s)} + \epsilon_t.
\]

Unlike the AA-positive control, \(A^{\ast}\) is **not constrained to be
nonnegative or to sum to one**. Thus the metastable regime is not a literal
simplex generator.


In [ ]:

def sample_state_sequence(
    T,
    K,
    rng,
    mean_dwell=MEAN_DWELL,
):
    states = []
    current = int(
        rng.integers(K)
    )

    while len(states) < T:
        dwell = max(
            2,
            int(
                rng.geometric(
                    1.0
                    / mean_dwell
                )
            ),
        )

        states.extend(
            [current] * dwell
        )

        candidates = [
            k
            for k in range(K)
            if k != current
        ]

        current = int(
            rng.choice(
                candidates
            )
        )

    return np.asarray(
        states[:T],
        dtype=int,
    )


def smooth_state_activations(
    states,
    K,
    transition_width=(
        TRANSITION_WIDTH
    ),
):
    T = len(states)

    A = np.zeros(
        (T, K),
        dtype=float,
    )

    A[
        np.arange(T),
        states,
    ] = 1.0

    if transition_width > 0:
        kernel_x = np.arange(
            -transition_width,
            transition_width + 1,
        )

        sigma = max(
            1.0,
            transition_width / 2,
        )

        kernel = np.exp(
            -(
                kernel_x ** 2
            )
            / (
                2
                * sigma ** 2
            )
        )

        kernel /= kernel.sum()

        for k in range(K):
            A[:, k] = np.convolve(
                A[:, k],
                kernel,
                mode="same",
            )

    A /= (
        A.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return A


def ar1_process(
    T,
    K,
    rho,
    rng,
):
    x = np.zeros(
        (T, K),
        dtype=float,
    )

    eps = rng.normal(
        0,
        1,
        size=(T, K),
    )

    x[0] = eps[0]

    innovation_scale = np.sqrt(
        max(
            1e-12,
            1.0 - rho ** 2,
        )
    )

    for t in range(
        1,
        T,
    ):
        x[t] = (
            rho * x[t - 1]
            + innovation_scale
            * eps[t]
        )

    return x


def add_correlated_secondary_recruitment(
    A,
    rng,
    secondary_weight=(
        SECONDARY_WEIGHT
    ),
    ar1_rho=AR1_RHO,
):
    T, K = A.shape

    background = ar1_process(
        T,
        K,
        ar1_rho,
        rng,
    )

    background = np.exp(
        background
    )

    background /= (
        background.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    mixed = (
        (
            1.0
            - secondary_weight
        )
        * A
        + secondary_weight
        * background
    )

    mixed /= (
        mixed.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return mixed


def add_non_simplex_modulation(
    A,
    rng,
    global_gain_sd=(
        GLOBAL_GAIN_SD
    ),
    signed_deviation_sd=(
        SIGNED_DEVIATION_SD
    ),
    deviation_rho=(
        DEVIATION_RHO
    ),
):
    T, K = A.shape

    gain_latent = ar1_process(
        T,
        1,
        deviation_rho,
        rng,
    )[:, 0]

    gain = np.exp(
        global_gain_sd
        * gain_latent
        - 0.5
        * global_gain_sd ** 2
    )

    signed_dev = ar1_process(
        T,
        K,
        deviation_rho,
        rng,
    )

    A_eff = (
        gain[:, None] * A
        + signed_deviation_sd
        * signed_dev
    )

    # Deliberately no nonnegative projection
    # and no row-sum normalization here.
    return A_eff, gain


def simulate_metastable(
    seed,
    noise_sd=BASE_NOISE_SD,
    heterogeneity_scale=1.0,
    motifs_raw=None,
    mean_dwell=MEAN_DWELL,
    transition_width=(
        TRANSITION_WIDTH
    ),
    secondary_weight=(
        SECONDARY_WEIGHT
    ),
    ar1_rho=AR1_RHO,
):
    rng = np.random.default_rng(
        seed
    )

    if motifs_raw is None:
        motifs_raw = (
            TRUE_MOTIFS_RAW
        )

    motifs_raw = np.asarray(
        motifs_raw,
        dtype=float,
    )

    subjects = []
    state_sequences = []
    activations = []
    simplex_activations = []
    gains = []

    for s in range(
        N_SUBJECTS
    ):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD
                * heterogeneity_scale,
                size=motifs_raw.shape,
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD
            * heterogeneity_scale,
            size=motifs_raw.shape,
        )

        subj_motifs = (
            motifs_raw
            * multiplicative
            + additive
        )

        states = (
            sample_state_sequence(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
                rng,
                mean_dwell=(
                    mean_dwell
                ),
            )
        )

        A_state = (
            smooth_state_activations(
                states,
                N_TRUE_MOTIFS,
                transition_width=(
                    transition_width
                ),
            )
        )

        A_simplex = (
            add_correlated_secondary_recruitment(
                A_state,
                rng,
                secondary_weight=(
                    secondary_weight
                ),
                ar1_rho=(
                    ar1_rho
                ),
            )
        )

        (
            A_eff,
            gain,
        ) = (
            add_non_simplex_modulation(
                A_simplex,
                rng,
            )
        )

        X = (
            A_eff @ subj_motifs
        )

        X += rng.normal(
            0,
            noise_sd,
            size=X.shape,
        )

        subjects.append(X)
        state_sequences.append(
            states
        )
        activations.append(
            A_eff
        )
        simplex_activations.append(
            A_simplex
        )
        gains.append(gain)

    return {
        "subjects": subjects,
        "true_patterns": motifs_raw,
        "states": state_sequences,
        "activations": activations,
        "simplex_activations": (
            simplex_activations
        ),
        "gains": gains,
        "truth_kind": "point",
        "dynamics": {
            "mean_dwell": (
                mean_dwell
            ),
            "transition_width": (
                transition_width
            ),
            "secondary_weight": (
                secondary_weight
            ),
            "ar1_rho": (
                ar1_rho
            ),
        },
    }


## 5. Metastable-process diagnostics

In [ ]:

def metastable_diagnostics(
    sim,
):
    dwell_lengths = []

    for states in sim["states"]:
        start = 0

        for t in range(
            1,
            len(states) + 1,
        ):
            if (
                t == len(states)
                or states[t]
                != states[start]
            ):
                dwell_lengths.append(
                    t - start
                )

                start = t

    lag1 = []

    for A in sim[
        "activations"
    ]:
        for k in range(
            A.shape[1]
        ):
            lag1.append(
                np.corrcoef(
                    A[:-1, k],
                    A[1:, k],
                )[0, 1]
            )

    all_A = np.vstack(
        sim["activations"]
    )

    row_sums = all_A.sum(
        axis=1
    )

    return {
        "mean_dwell": float(
            np.mean(
                dwell_lengths
            )
        ),
        "median_dwell": float(
            np.median(
                dwell_lengths
            )
        ),
        "mean_activation_lag1": float(
            np.nanmean(
                lag1
            )
        ),
        "activation_row_sum_sd": float(
            np.std(
                row_sums
            )
        ),
        "fraction_negative_coefficients": float(
            np.mean(
                all_A < 0
            )
        ),
    }


demo = simulate_metastable(
    RNG_SEED
)

print(
    metastable_diagnostics(
        demo
    )
)

fig, ax = plt.subplots(
    figsize=(10, 4)
)

ax.plot(
    demo["activations"][0]
)

ax.set_xlabel("Time")
ax.set_ylabel(
    "Motif activation"
)

ax.set_title(
    "Example non-simplex metastable activation trajectories"
)

plt.show()


## 6. Restart-safe checkpoints

In [ ]:

def load_results():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        print(
            f"Loaded {len(df)} completed simulations"
        )
        return df

    return pd.DataFrame()


def is_complete(
    df,
    regime,
    condition,
    rep,
):
    if df.empty:
        return False

    mask = (
        (df["regime"] == regime)
        & (df["condition"] == condition)
        & (
            df["rep"].astype(int)
            == int(rep)
        )
    )

    return bool(mask.any())


def save_row(df, row):
    new = pd.DataFrame([row])

    out = (
        new
        if df.empty
        else pd.concat(
            [df, new],
            ignore_index=True,
        )
    )

    out = out.drop_duplicates(
        subset=[
            "regime",
            "condition",
            "rep",
        ],
        keep="last",
    )

    tmp = RESULTS_CSV.with_suffix(
        ".tmp.csv"
    )

    out.to_csv(
        tmp,
        index=False,
    )

    os.replace(
        tmp,
        RESULTS_CSV,
    )

    return out


# PART A — Four-regime core benchmark

In [ ]:

results_df = load_results()

core_specs = [
    (
        "PCA_control",
        simulate_pca_control,
        N_REPS_CORE,
        101,
    ),
    (
        "ICA_control",
        simulate_ica_control,
        N_REPS_CORE,
        151,
    ),
    (
        "AA_control",
        simulate_aa_control,
        N_REPS_CORE,
        202,
    ),
]

for (
    regime,
    generator,
    n_reps,
    regime_code,
) in core_specs:

    for rep in range(
        n_reps
    ):
        condition = "base"

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {regime} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    regime_code,
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = generator(seed)

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=(
                N_TRUE_MOTIFS
            ),
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                )
            },
        )

        results_df = save_row(
            results_df,
            row,
        )


for rep in range(
    N_REPS_METASTABLE
):
    regime = "metastable"
    condition = "base"

    if is_complete(
        results_df,
        regime,
        condition,
        rep,
    ):
        print(
            f"SKIP {regime} "
            f"rep={rep}"
        )
        continue

    seed = int(
        np.random.SeedSequence(
            [
                RNG_SEED,
                303,
                rep,
            ]
        ).generate_state(1)[0]
    )

    sim = simulate_metastable(
        seed
    )

    diag = (
        metastable_diagnostics(
            sim
        )
    )

    row = evaluate_methods(
        sim["subjects"],
        sim["true_patterns"],
        K_fit=N_TRUE_MOTIFS,
        seed=seed,
        regime=regime,
        rep=rep,
        true_activations=(
            sim["activations"]
        ),
        truth_kind=(
            sim["truth_kind"]
        ),
        extra={
            "condition": condition,
            **diag,
        },
    )

    results_df = save_row(
        results_df,
        row,
    )

print(
    "Core four-regime benchmark complete."
)


## 7. Core summary

In [ ]:

core = results_df[
    results_df["regime"].isin(
        [
            "PCA_control",
            "ICA_control",
            "AA_control",
            "metastable",
        ]
    )
].copy()

summary_cols = [
    "aa_motif_recovery",
    "pca_motif_recovery",
    "ica_motif_recovery",
    "aa_temporal_recovery",
    "pca_temporal_recovery",
    "ica_temporal_recovery",
    "aa_subnetwork_recovery",
    "pca_subnetwork_recovery",
    "ica_subnetwork_recovery",
    "aa_span_recovery",
    "pca_span_recovery",
    "ica_span_recovery",
    "aa_mse",
    "pca_mse",
    "ica_mse",
]

summary = (
    core
    .groupby(
        "regime"
    )[
        summary_cols
    ]
    .agg(
        ["mean", "sem"]
    )
)

display(summary)


In [ ]:

# Compact manuscript-oriented means table

compact = (
    core
    .groupby("regime")
    .agg(
        n=("rep", "count"),

        AA_spatial=(
            "aa_motif_recovery",
            "mean",
        ),
        PCA_spatial=(
            "pca_motif_recovery",
            "mean",
        ),
        ICA_spatial=(
            "ica_motif_recovery",
            "mean",
        ),

        AA_temporal=(
            "aa_temporal_recovery",
            "mean",
        ),
        PCA_temporal=(
            "pca_temporal_recovery",
            "mean",
        ),
        ICA_temporal=(
            "ica_temporal_recovery",
            "mean",
        ),

        AA_subnetwork=(
            "aa_subnetwork_recovery",
            "mean",
        ),
        PCA_subnetwork=(
            "pca_subnetwork_recovery",
            "mean",
        ),
        ICA_subnetwork=(
            "ica_subnetwork_recovery",
            "mean",
        ),

        AA_span=(
            "aa_span_recovery",
            "mean",
        ),
        PCA_span=(
            "pca_span_recovery",
            "mean",
        ),
        ICA_span=(
            "ica_span_recovery",
            "mean",
        ),

        AA_MSE=(
            "aa_mse",
            "mean",
        ),
        PCA_MSE=(
            "pca_mse",
            "mean",
        ),
        ICA_MSE=(
            "ica_mse",
            "mean",
        ),

        ICA_convergence_rate=(
            "ica_converged",
            "mean",
        ),
    )
)

display(compact)


## 8. Core figure: direct pattern recovery

In [ ]:

regime_order = [
    "PCA_control",
    "ICA_control",
    "AA_control",
    "metastable",
]

regime_labels = [
    "PCA-favorable",
    "ICA-favorable",
    "AA-favorable",
    "Metastable",
]

method_cols = [
    ("MS-AA", "aa_motif_recovery"),
    ("PCA", "pca_motif_recovery"),
    ("ICA", "ica_motif_recovery"),
]

x = np.arange(
    len(regime_order)
)

width = 0.24

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

for j, (label, col) in enumerate(
    method_cols
):
    means = []
    sems = []

    for regime in regime_order:
        vals = core.loc[
            core["regime"] == regime,
            col,
        ]

        means.append(vals.mean())
        sems.append(vals.sem())

    offset = (
        j - 1
    ) * width

    ax.bar(
        x + offset,
        means,
        width,
        yerr=sems,
        label=label,
        capsize=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(regime_labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel(
    "One-to-one true-pattern recovery"
)
ax.set_title(
    "Method-calibrated latent-pattern recovery"
)
ax.legend()

plt.show()


## 9. Span-recovery control

In [ ]:

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

span_cols = [
    ("MS-AA", "aa_span_recovery"),
    ("PCA", "pca_span_recovery"),
    ("ICA", "ica_span_recovery"),
]

for j, (label, col) in enumerate(
    span_cols
):
    means = []
    sems = []

    for regime in regime_order:
        vals = core.loc[
            core["regime"] == regime,
            col,
        ]

        means.append(vals.mean())
        sems.append(vals.sem())

    offset = (
        j - 1
    ) * width

    ax.bar(
        x + offset,
        means,
        width,
        yerr=sems,
        label=label,
        capsize=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(regime_labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel(
    "Recovery from full recovered span"
)
ax.set_title(
    "Pattern identity versus low-dimensional span"
)
ax.legend()

plt.show()


# PART B — Paired bootstrap and sign-flip inference

In [ ]:

PAIR_BOOT_REPS = 20000
PAIR_PERM_REPS = 20000
PAIR_RNG_SEED = 20260908


def bootstrap_mean_ci(
    values,
    n_boot=PAIR_BOOT_REPS,
    seed=PAIR_RNG_SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(values),
        size=(
            n_boot,
            len(values),
        ),
    )

    boot_means = values[idx].mean(
        axis=1
    )

    lo, hi = np.quantile(
        boot_means,
        [0.025, 0.975],
    )

    return (
        float(values.mean()),
        float(lo),
        float(hi),
    )


def signflip_pvalue(
    values,
    n_perm=PAIR_PERM_REPS,
    seed=PAIR_RNG_SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    observed = abs(
        values.mean()
    )

    rng = np.random.default_rng(seed)

    signs = rng.choice(
        [-1.0, 1.0],
        size=(
            n_perm,
            len(values),
        ),
    )

    perm_means = np.abs(
        (
            signs
            * values[None, :]
        ).mean(axis=1)
    )

    p = (
        np.sum(
            perm_means >= observed
        )
        + 1
    ) / (
        n_perm + 1
    )

    return float(p)


comparison_specs = [
    (
        "AA - PCA",
        "aa_motif_recovery",
        "pca_motif_recovery",
    ),
    (
        "AA - ICA",
        "aa_motif_recovery",
        "ica_motif_recovery",
    ),
    (
        "ICA - PCA",
        "ica_motif_recovery",
        "pca_motif_recovery",
    ),
]

paired_rows = []

for r_i, regime in enumerate(
    regime_order
):
    sub = (
        core[
            core["regime"] == regime
        ]
        .sort_values("rep")
        .copy()
    )

    for c_i, (
        comparison,
        col_a,
        col_b,
    ) in enumerate(comparison_specs):

        delta = (
            sub[col_a].to_numpy()
            - sub[col_b].to_numpy()
        )

        mean, lo, hi = (
            bootstrap_mean_ci(
                delta,
                seed=(
                    PAIR_RNG_SEED
                    + 100 * r_i
                    + c_i
                ),
            )
        )

        paired_rows.append({
            "regime": regime,
            "comparison": comparison,
            "n": len(delta),
            "mean_delta": mean,
            "CI_low": lo,
            "CI_high": hi,
            "signflip_p": (
                signflip_pvalue(
                    delta,
                    seed=(
                        PAIR_RNG_SEED
                        + 1000
                        + 100 * r_i
                        + c_i
                    ),
                )
            ),
            "fraction_positive": float(
                np.mean(delta > 0)
            ),
        })

paired_inference_df = pd.DataFrame(
    paired_rows
)

display(paired_inference_df)


## 10. Metastable pairwise method differences

In [ ]:

meta_pairs = paired_inference_df[
    paired_inference_df["regime"]
    == "metastable"
].copy()

display(meta_pairs)

fig, ax = plt.subplots(
    figsize=(7.5, 4.5)
)

x_meta = np.arange(
    len(meta_pairs)
)

means = meta_pairs[
    "mean_delta"
].to_numpy()

los = meta_pairs[
    "CI_low"
].to_numpy()

his = meta_pairs[
    "CI_high"
].to_numpy()

ax.errorbar(
    x_meta,
    means,
    yerr=np.vstack([
        means - los,
        his - means,
    ]),
    fmt="o",
    capsize=5,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(x_meta)
ax.set_xticklabels(
    meta_pairs["comparison"]
)
ax.set_ylabel(
    "Paired difference in motif recovery"
)
ax.set_title(
    "Brain-inspired metastable regime"
)

plt.show()


# PART C — Metastable robustness to noise

In [ ]:

results_df = load_results()

for noise_sd in NOISE_LEVELS:
    for rep in range(
        N_REPS_ROBUST
    ):
        regime = (
            "metastable_noise"
        )

        condition = (
            f"noise_{noise_sd}"
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    404,
                    int(
                        noise_sd
                        * 1000
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=noise_sd,
            heterogeneity_scale=1.0,
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                "noise_sd": (
                    noise_sd
                ),
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Noise robustness complete."
)


# PART D — Metastable robustness to subject heterogeneity

In [ ]:

results_df = load_results()

for hetero in (
    HETEROGENEITY_LEVELS
):
    for rep in range(
        N_REPS_ROBUST
    ):
        regime = (
            "metastable_heterogeneity"
        )

        condition = (
            f"heterogeneity_{hetero}"
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    505,
                    int(
                        hetero
                        * 1000
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=(
                BASE_NOISE_SD
            ),
            heterogeneity_scale=(
                hetero
            ),
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                "heterogeneity_scale": (
                    hetero
                ),
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Heterogeneity robustness complete."
)


# PART E — Metastable sensitivity to fitted K

In [ ]:

results_df = load_results()

for K_fit in K_LEVELS:
    for rep in range(
        N_REPS_ROBUST
    ):
        regime = "metastable_K"

        condition = (
            f"K_{K_fit}"
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    606,
                    int(K_fit),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = (
            simulate_metastable(
                seed
            )
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=K_fit,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "K sensitivity complete."
)



## K-sensitivity summary

For \(K < K_{\mathrm{true}}\), use the **coverage-adjusted** score as
the primary quantity. The ordinary matched mean is retained only as a
diagnostic.

Temporal recovery also receives a coverage-adjusted version so that
recovering only a subset of the six latent configurations cannot look
artificially strong.


In [ ]:

k_df = results_df[
    results_df["regime"]
    == "metastable_K"
].copy()

if len(k_df):
    k_summary = (
        k_df
        .groupby("K_fit")[
            [
                "aa_motif_recovery_coverage",
                "pca_motif_recovery_coverage",
                "ica_motif_recovery_coverage",
                "aa_temporal_recovery_coverage",
                "pca_temporal_recovery_coverage",
                "ica_temporal_recovery_coverage",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )

    display(k_summary)

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for label, col in [
        (
            "MS-AA",
            "aa_motif_recovery_coverage",
        ),
        (
            "PCA",
            "pca_motif_recovery_coverage",
        ),
        (
            "ICA",
            "ica_motif_recovery_coverage",
        ),
    ]:
        means = (
            k_df
            .groupby(
                "K_fit"
            )[col]
            .mean()
        )

        sems = (
            k_df
            .groupby(
                "K_fit"
            )[col]
            .sem()
        )

        ax.errorbar(
            means.index,
            means.values,
            yerr=sems.values,
            marker="o",
            label=label,
        )

    ax.axvline(
        N_TRUE_MOTIFS,
        linestyle="--",
        linewidth=1,
    )

    ax.set_xlabel(
        "Fitted K"
    )

    ax.set_ylabel(
        "Coverage-adjusted spatial recovery"
    )

    ax.set_title(
        "Metastable sensitivity to fitted model order"
    )

    ax.legend()

    plt.show()



# PART F — Metastable sensitivity to motif overlap

The same metastable dynamics are generated from three versions of the
distributed motifs:

- **low overlap**: motifs are made more distinct,
- **baseline**: original motif geometry,
- **high overlap**: motifs are blended toward the common distributed pattern.

The notebook prints the mean pairwise cosine similarity of the true motifs so
the manipulation can be verified directly.


In [ ]:

results_df = load_results()

for (
    overlap_name,
    distinctiveness_scale,
) in OVERLAP_LEVELS.items():

    motifs_variant = (
        make_overlap_variant(
            TRUE_MOTIFS_RAW,
            distinctiveness_scale,
        )
    )

    overlap_cosine = (
        mean_pairwise_cosine(
            motifs_variant
        )
    )

    print(
        overlap_name,
        "| mean pairwise cosine =",
        round(
            overlap_cosine,
            3,
        ),
    )

    for rep in range(
        N_REPS_SENSITIVITY
    ):
        regime = (
            "metastable_overlap"
        )

        condition = (
            overlap_name
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    707,
                    list(
                        OVERLAP_LEVELS
                    ).index(
                        overlap_name
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = (
            simulate_metastable(
                seed,
                motifs_raw=(
                    motifs_variant
                ),
            )
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                "distinctiveness_scale": (
                    distinctiveness_scale
                ),
                "mean_pairwise_cosine": (
                    overlap_cosine
                ),
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Overlap sensitivity complete."
)


In [ ]:

overlap_df = results_df[
    results_df["regime"]
    == "metastable_overlap"
].copy()

if len(overlap_df):
    display(
        overlap_df
        .groupby(
            "condition"
        )[
            [
                "mean_pairwise_cosine",
                "aa_motif_recovery",
                "pca_motif_recovery",
                "ica_motif_recovery",
                "aa_temporal_recovery",
                "pca_temporal_recovery",
                "ica_temporal_recovery",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )

    order = [
        "low_overlap",
        "baseline_overlap",
        "high_overlap",
    ]

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for label, col in [
        (
            "MS-AA",
            "aa_motif_recovery",
        ),
        (
            "PCA",
            "pca_motif_recovery",
        ),
        (
            "ICA",
            "ica_motif_recovery",
        ),
    ]:
        means = [
            overlap_df.loc[
                overlap_df[
                    "condition"
                ] == cond,
                col,
            ].mean()
            for cond in order
        ]

        sems = [
            overlap_df.loc[
                overlap_df[
                    "condition"
                ] == cond,
                col,
            ].sem()
            for cond in order
        ]

        ax.errorbar(
            np.arange(
                len(order)
            ),
            means,
            yerr=sems,
            marker="o",
            label=label,
        )

    ax.set_xticks(
        np.arange(
            len(order)
        )
    )

    ax.set_xticklabels(
        [
            "Low",
            "Baseline",
            "High",
        ]
    )

    ax.set_xlabel(
        "Motif overlap"
    )

    ax.set_ylabel(
        "Spatial motif recovery"
    )

    ax.set_title(
        "Sensitivity to latent motif overlap"
    )

    ax.legend()

    plt.show()



# PART G — Metastable sensitivity to dynamical mixing

This is a deliberately small, interpretable sensitivity analysis rather than a
large factorial sweep.

- **state-like**: longer dwell times, sharper transitions, less secondary
  recruitment;
- **baseline**: manuscript default;
- **more mixed**: shorter dwell times, broader transitions, stronger
  secondary recruitment, and more persistent background activity.

The question is whether the result disappears once configurations become less
discrete and more continuously blended.


In [ ]:

results_df = load_results()

for (
    dynamics_name,
    pars,
) in DYNAMICS_PRESETS.items():

    for rep in range(
        N_REPS_SENSITIVITY
    ):
        regime = (
            "metastable_dynamics"
        )

        condition = (
            dynamics_name
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} "
                f"rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    808,
                    list(
                        DYNAMICS_PRESETS
                    ).index(
                        dynamics_name
                    ),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            mean_dwell=(
                pars[
                    "mean_dwell"
                ]
            ),
            transition_width=(
                pars[
                    "transition_width"
                ]
            ),
            secondary_weight=(
                pars[
                    "secondary_weight"
                ]
            ),
            ar1_rho=(
                pars[
                    "ar1_rho"
                ]
            ),
        )

        diag = (
            metastable_diagnostics(
                sim
            )
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            true_activations=(
                sim["activations"]
            ),
            truth_kind=(
                sim["truth_kind"]
            ),
            extra={
                "condition": (
                    condition
                ),
                **pars,
                **diag,
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print(
    "Dynamics sensitivity complete."
)


In [ ]:

dynamics_df = results_df[
    results_df["regime"]
    == "metastable_dynamics"
].copy()

if len(dynamics_df):
    display(
        dynamics_df
        .groupby(
            "condition"
        )[
            [
                "mean_dwell",
                "mean_activation_lag1",
                "aa_motif_recovery",
                "pca_motif_recovery",
                "ica_motif_recovery",
                "aa_temporal_recovery",
                "pca_temporal_recovery",
                "ica_temporal_recovery",
            ]
        ]
        .agg(
            ["mean", "sem"]
        )
    )

    order = [
        "state_like",
        "baseline",
        "more_mixed",
    ]

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for label, col in [
        (
            "MS-AA",
            "aa_motif_recovery",
        ),
        (
            "PCA",
            "pca_motif_recovery",
        ),
        (
            "ICA",
            "ica_motif_recovery",
        ),
    ]:
        means = [
            dynamics_df.loc[
                dynamics_df[
                    "condition"
                ] == cond,
                col,
            ].mean()
            for cond in order
        ]

        sems = [
            dynamics_df.loc[
                dynamics_df[
                    "condition"
                ] == cond,
                col,
            ].sem()
            for cond in order
        ]

        ax.errorbar(
            np.arange(
                len(order)
            ),
            means,
            yerr=sems,
            marker="o",
            label=label,
        )

    ax.set_xticks(
        np.arange(
            len(order)
        )
    )

    ax.set_xticklabels(
        [
            "State-like",
            "Baseline",
            "More mixed",
        ]
    )

    ax.set_ylabel(
        "Spatial motif recovery"
    )

    ax.set_title(
        "Sensitivity to dynamical mixing"
    )

    ax.legend()

    plt.show()



# PART H — Spatial versus temporal recovery

The spatial analysis asks **what** configuration was recovered. The temporal
analysis asks whether the **same spatially matched component/archetype**
recovered **when** that configuration was expressed.

For PCA and ICA, temporal sign is ignored because component signs are
arbitrary. MS-AA is evaluated without sign reversal.


In [ ]:

core = results_df[
    results_df["regime"].isin(
        regime_order
    )
].copy()

temporal_summary = (
    core
    .groupby("regime")[
        [
            "aa_temporal_recovery",
            "pca_temporal_recovery",
            "ica_temporal_recovery",
        ]
    ]
    .agg(
        ["mean", "sem"]
    )
)

display(
    temporal_summary
)

meta_temporal = core[
    core["regime"]
    == "metastable"
].copy()

temporal_pairs = []

for i, (
    label,
    col_a,
    col_b,
) in enumerate([
    (
        "AA - PCA",
        "aa_temporal_recovery",
        "pca_temporal_recovery",
    ),
    (
        "AA - ICA",
        "aa_temporal_recovery",
        "ica_temporal_recovery",
    ),
    (
        "ICA - PCA",
        "ica_temporal_recovery",
        "pca_temporal_recovery",
    ),
]):
    delta = (
        meta_temporal[
            col_a
        ].to_numpy()
        - meta_temporal[
            col_b
        ].to_numpy()
    )

    mean, lo, hi = (
        bootstrap_mean_ci(
            delta,
            seed=(
                PAIR_RNG_SEED
                + 3000
                + i
            ),
        )
    )

    temporal_pairs.append({
        "comparison": label,
        "n": len(delta),
        "mean_delta": mean,
        "CI_low": lo,
        "CI_high": hi,
        "signflip_p": (
            signflip_pvalue(
                delta,
                seed=(
                    PAIR_RNG_SEED
                    + 4000
                    + i
                ),
            )
        ),
        "fraction_positive": float(
            np.mean(
                delta > 0
            )
        ),
    })

temporal_pair_df = (
    pd.DataFrame(
        temporal_pairs
    )
)

display(
    temporal_pair_df
)



# PART I — Normalization QA

The empirical fitting notebook first centers each subject feature-wise and then
uses `normalize(..., normalize="across")`. The simulation fallback follows the
same order.

If `hypertools/normalize.py` is available in the current environment, this
cell compares the exact empirical routine against the fallback on simulated
data. A very small maximum absolute difference supports equivalence. If the
file is unavailable, the notebook prints a message and continues.


In [ ]:

def load_empirical_normalize_module(path):
    """
    Load the empirical hypertools/normalize.py module even though it uses
    local imports such as:

        from format_data import format_data as formatter
        from helpers import memoize

    Loading normalize.py directly with spec_from_file_location() does not
    automatically put its containing directory on sys.path. We therefore
    expose that directory temporarily while the module and its local
    dependencies are imported.
    """
    path = Path(path).expanduser().resolve()

    if not path.exists():
        raise FileNotFoundError(
            f"Empirical normalize.py not found: {path}"
        )

    module_dir = str(path.parent)
    added_path = module_dir not in sys.path

    if added_path:
        sys.path.insert(
            0,
            module_dir,
        )

    try:
        spec = importlib.util.spec_from_file_location(
            "empirical_normalize",
            str(path),
        )

        if spec is None or spec.loader is None:
            raise ImportError(
                f"Could not create import spec for {path}"
            )

        ht = importlib.util.module_from_spec(
            spec
        )

        spec.loader.exec_module(
            ht
        )

    finally:
        if added_path:
            try:
                sys.path.remove(
                    module_dir
                )
            except ValueError:
                pass

    return ht


def normalization_qa():
    """
    Compare the simulation fallback 'across' normalization with the exact
    empirical hypertools implementation on the same simulated subject data.

    This QA does not change the simulation results automatically. It reports
    the maximum absolute discrepancy so that we can decide whether the
    simulation normalizer needs to be replaced by the empirical routine.
    """
    qa_sim = simulate_metastable(
        123456
    )

    raw = qa_sim[
        "subjects"
    ][:3]

    fallback, _, _ = (
        normalize_across_subjects(
            raw
        )
    )

    normalize_path = (
        Path(
            HYPERTOOLS_NORMALIZE_PATH
        )
        .expanduser()
        .resolve()
    )

    if not normalize_path.exists():
        print(
            "Exact hypertools normalizer not found at:",
            normalize_path,
        )
        print(
            "Normalization QA skipped; this does not affect the simulation run."
        )
        return None

    ht = load_empirical_normalize_module(
        normalize_path
    )

    coerced = [
        coerce_TxV(x)
        for x in raw
    ]

    # We have already converted the inputs to numeric T x V arrays.
    # Disable HyperTools' separate format_data() layer here; otherwise
    # normalize() tries to import optional legacy dependencies such as
    # df2mat/text2mat/datageometry that are irrelevant to this QA.
    exact = ht.normalize(
        coerced,
        normalize="across",
        format_data=False,
    )

    diffs = [
        np.max(
            np.abs(
                np.asarray(
                    a,
                    dtype=float,
                )
                - np.asarray(
                    b,
                    dtype=float,
                )
            )
        )
        for a, b in zip(
            fallback,
            exact,
        )
    ]

    max_diff = float(
        np.max(diffs)
    )

    print(
        "Maximum absolute fallback-vs-exact difference:",
        max_diff,
    )

    if np.isfinite(max_diff):
        if max_diff < 1e-10:
            print(
                "Normalization QA: PASS — fallback and empirical "
                "normalization agree to numerical precision."
            )
        elif max_diff < 1e-6:
            print(
                "Normalization QA: very small numerical discrepancy."
            )
        else:
            print(
                "Normalization QA: substantive discrepancy detected. "
                "Inspect the empirical normalization definition before "
                "treating the simulation normalization as pipeline-matched."
            )

    return max_diff


NORMALIZATION_QA_MAX_DIFF = (
    normalization_qa()
)


# PART J — Representative metastable spatial and temporal recovery

In [ ]:

REPRESENTATIVE_SEED = 20260908

sim = simulate_metastable(
    REPRESENTATIVE_SEED
)

(
    subjects,
    mean_subject_mean,
    sd,
) = normalize_across_subjects(
    sim["subjects"]
)

true_patterns = (
    transform_true_patterns(
        sim["true_patterns"],
        mean_subject_mean,
        sd,
        truth_kind=(
            sim["truth_kind"]
        ),
    )
)

X_stack = np.vstack(
    subjects
)

# ---------- MS-AA ----------
msaa = fit_msaa(
    subjects,
    K=N_TRUE_MOTIFS,
    analysis_type="spatial",
    seed=REPRESENTATIVE_SEED,
    verbose=True,
)

S_group = np.mean(
    np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            )
            for sub in (
                msaa[
                    "results_subj"
                ]
            )
        ],
        axis=0,
    ),
    axis=0,
)

aa_scores = [
    np.asarray(
        sub["sXC"],
        dtype=float,
    )
    for sub in (
        msaa[
            "results_subj"
        ]
    )
]

# ---------- PCA ----------
pca = PCA(
    n_components=N_TRUE_MOTIFS,
    random_state=(
        REPRESENTATIVE_SEED
    ),
).fit(
    X_stack
)

pca_patterns = (
    pca.components_
)

pca_scores = (
    pca.transform(
        X_stack
    )
)

# ---------- ICA ----------
ica_fit = fit_fastica(
    X_stack,
    N_TRUE_MOTIFS,
    REPRESENTATIVE_SEED,
)

ica_patterns = (
    ica_fit[
        "spatial_patterns"
    ]
)

ica_scores = (
    ica_fit["scores"]
)

print(
    "ICA converged:",
    ica_fit["converged"],
    "| iterations:",
    ica_fit["n_iter"],
)

# ---------- Spatial matching ----------
aa_match = optimal_match(
    true_patterns,
    S_group,
    absolute=False,
)

pca_match = optimal_match(
    true_patterns,
    pca_patterns,
    absolute=True,
)

ica_match = optimal_match(
    true_patterns,
    ica_patterns,
    absolute=True,
)

AA_ORDERED = S_group[
    aa_match[
        "recovered_idx"
    ]
].copy()

PCA_ORDERED = (
    pca_patterns[
        pca_match[
            "recovered_idx"
        ]
    ].copy()
)

ICA_ORDERED = (
    ica_patterns[
        ica_match[
            "recovered_idx"
        ]
    ].copy()
)

# Orient PCA/ICA signs for display.
for i in range(
    min(
        len(PCA_ORDERED),
        len(true_patterns),
    )
):
    if (
        np.corrcoef(
            true_patterns[i],
            PCA_ORDERED[i],
        )[0, 1]
        < 0
    ):
        PCA_ORDERED[i] *= -1

for i in range(
    min(
        len(ICA_ORDERED),
        len(true_patterns),
    )
):
    if (
        np.corrcoef(
            true_patterns[i],
            ICA_ORDERED[i],
        )[0, 1]
        < 0
    ):
        ICA_ORDERED[i] *= -1

plot_pattern_matrix(
    true_patterns,
    "True recurring network configurations",
    ylabel=(
        "True configuration"
    ),
)

plot_pattern_matrix(
    AA_ORDERED,
    "Matched MS-AA archetypes",
    ylabel=(
        "Matched archetype"
    ),
)

plot_pattern_matrix(
    PCA_ORDERED,
    "Matched PCA components",
    ylabel=(
        "Matched component"
    ),
)

plot_pattern_matrix(
    ICA_ORDERED,
    "Matched ICA spatial mixing patterns",
    ylabel=(
        "Matched ICA pattern"
    ),
)

# ---------- Temporal recovery ----------
aa_temp = (
    temporal_recovery_from_spatial_match(
        sim["activations"],
        aa_scores,
        aa_match,
        absolute=False,
    )
)

pca_temp = (
    temporal_recovery_from_spatial_match(
        sim["activations"],
        pca_scores,
        pca_match,
        absolute=True,
    )
)

ica_temp = (
    temporal_recovery_from_spatial_match(
        sim["activations"],
        ica_scores,
        ica_match,
        absolute=True,
    )
)

print(
    "Representative temporal recovery:",
    {
        "MS-AA": aa_temp["mean"],
        "PCA": pca_temp["mean"],
        "ICA": ica_temp["mean"],
    },
)


# PART K — Final manuscript-oriented summary

In [ ]:

results_df = load_results()

core = results_df[
    results_df["regime"].isin(
        regime_order
    )
].copy()

final_rows = []

for regime in regime_order:
    sub = core[
        core["regime"]
        == regime
    ]

    if len(sub) == 0:
        continue

    final_rows.append({
        "regime": regime,
        "n": len(sub),

        "MS-AA spatial": (
            sub[
                "aa_motif_recovery"
            ].mean()
        ),
        "PCA spatial": (
            sub[
                "pca_motif_recovery"
            ].mean()
        ),
        "ICA spatial": (
            sub[
                "ica_motif_recovery"
            ].mean()
        ),

        "MS-AA temporal": (
            sub[
                "aa_temporal_recovery"
            ].mean()
        ),
        "PCA temporal": (
            sub[
                "pca_temporal_recovery"
            ].mean()
        ),
        "ICA temporal": (
            sub[
                "ica_temporal_recovery"
            ].mean()
        ),

        "MS-AA subnetwork": (
            sub[
                "aa_subnetwork_recovery"
            ].mean()
        ),
        "PCA subnetwork": (
            sub[
                "pca_subnetwork_recovery"
            ].mean()
        ),
        "ICA subnetwork": (
            sub[
                "ica_subnetwork_recovery"
            ].mean()
        ),

        "MS-AA span": (
            sub[
                "aa_span_recovery"
            ].mean()
        ),
        "PCA span": (
            sub[
                "pca_span_recovery"
            ].mean()
        ),
        "ICA span": (
            sub[
                "ica_span_recovery"
            ].mean()
        ),

        "MS-AA MSE": (
            sub[
                "aa_mse"
            ].mean()
        ),
        "PCA MSE": (
            sub[
                "pca_mse"
            ].mean()
        ),
        "ICA MSE": (
            sub[
                "ica_mse"
            ].mean()
        ),

        "ICA convergence rate": (
            sub[
                "ica_converged"
            ].mean()
        ),
    })

final_summary_df = (
    pd.DataFrame(
        final_rows
    )
)

display(
    final_summary_df
)



# Interpretation guide

A well-calibrated benchmark should show **method-specific reversals** rather
than universal superiority.

### PCA-favorable control
PCA should recover the true covariance directions best.

### ICA-favorable control
ICA should most directly recover independent non-Gaussian mixing patterns.

### AA-favorable control
MS-AA should most directly recover extremal convex-mixture motifs.

### Brain-inspired metastable regime
This is the key scientific comparison. The generator contains persistence,
gradual transitions, coactivation, subject variability, noise, time-varying
gain, and signed autocorrelated deviations. It is therefore **not an exact
PCA, ICA, or AA generative model**.

The primary outcomes are:

1. **spatial motif recovery** — what recurring configuration was recovered?
2. **temporal recovery** — did that same matched dimension recover when the
   configuration was expressed?
3. **subnetwork recovery** — was the network composition preserved?
4. **span recovery** — was the latent structure retained even if individual
   basis vectors were rotated or mixed?
5. **reconstruction error** — how well was observed variance reconstructed?

A strong archetypal result does **not** require MS-AA to beat PCA or ICA on
reconstruction. The more specific hypothesis is that MS-AA can more directly
identify recurring distributed configurations under an overlapping,
metastable process, even when variance-based or independence-based methods
retain the same low-dimensional information.

### Sensitivity interpretation
The overlap and dynamics sweeps ask whether any MS-AA advantage depends on
a hand-picked, nearly discrete configuration geometry. The important result
would be persistence of the qualitative effect as configurations become more
overlapping and transitions/coactivation become more continuous.

### ICA convergence
ICA recovery values are retained for every dataset. Convergence warnings are
reported diagnostically rather than converted to missing values. This is
especially relevant for the Gaussian PCA-positive control, where independent
components are not uniquely identifiable.

### Caution
This simulation is evidence about **representational geometry**, not evidence
that real neural dynamics are literally simplex-organized or that recovered
archetypes are biological attractors.
